# Policy Stacks, Indicators, and Infeasibility

This notebook shows how **stacked policies** and logical constraints can make a model infeasible – and how to diagnose which rules are in conflict.

You will:
- Build a small **shift scheduling** model with several policy rules.
- Represent those rules in a **policy stack table** (decisions × policy categories).
- See a feasible solution when the rules are consistent.
- Add an extra rule that makes the model **infeasible**.
- Relax rules one at a time to identify which policy (or combination) causes the problem.

## Key Concepts

**Policy stack (constraint matrix)**
- Rows: decisions (e.g., which shifts to staff).
- Columns: policy categories (e.g., coverage, rest rules, senior coverage).
- A mark in a cell means that decision is subject to that policy.

**Infeasibility**
- No solution can satisfy all constraints simultaneously.
- Often caused by new policies that conflict with existing ones.

**Diagnosis**
- Remove or relax constraints one at a time.
- When feasibility returns, the relaxed rule is part of the conflict.

**Critical insight**: Infeasibility is not a model failure – it reveals a **policy conflict** that may need business resolution.

## Step 1: Install Required Packages

### Setup: Install Required Packages

To run this notebook, we need the PuLP optimization library and basic data/plotting libraries.

**What this code does:** Installs `pulp`, `pandas`, `numpy`, and `matplotlib`.

**What to look for in the output:** You should see "Requirement already satisfied" if these packages are already installed, or installation messages otherwise.

In [1]:
# Install required packages (needed in Google Colab; can be skipped locally if already installed)
%pip install pulp pandas numpy matplotlib -q


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 2: Import Libraries

### Setup: Import Libraries

We'll use these libraries to build our models and policy stack.

**What this code does:** Imports `pandas`, `numpy`, `matplotlib`, and `pulp`.

**What to look for in the output:** Successful imports run silently; errors indicate a missing package.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pulp import LpProblem, LpVariable, LpMinimize, LpStatus, lpSum, PULP_CBC_CMD

plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

## Step 3: Scenario – Shift Leads for a 3-Shift Operations Center

Consider a 24/7 operations center with three shifts:
- `Day`
- `Evening`
- `Night`

**Decisions (binary):**
- `x_Day`, `x_Evening`, `x_Night` – 1 if the shift is staffed, 0 if not.

**Policies (initial, consistent set):**
1. **Coverage**: At least two shifts must be staffed each day.
2. **Night dependency**: The night shift can only run if the evening shift is also staffed.
3. **Senior lead requirement**: If the night shift is staffed, at least one senior lead must be assigned.

We encode a simple version of these rules and confirm that the model is **feasible** before adding an over-tight rule.

In [3]:
# Decisions
shifts = ["Day", "Evening", "Night"]

# Simple costs to give the model something to minimize
costs = {"Day": 5, "Evening": 6, "Night": 7}  # relative cost units

# Policy descriptions for reference
policies = [
    "Coverage: at least two shifts staffed",
    "Night dependency: Night => Evening",
    "Senior lead: Night staffed => senior lead assigned",
]

print("SHIFTS AND COSTS:")
for s in shifts:
    print(f"  {s}: cost {costs[s]}")

print("\nPOLICIES (initial set):")
for p in policies:
    print(" -", p)

SHIFTS AND COSTS:
  Day: cost 5
  Evening: cost 6
  Night: cost 7

POLICIES (initial set):
 - Coverage: at least two shifts staffed
 - Night dependency: Night => Evening
 - Senior lead: Night staffed => senior lead assigned


## Step 4: Policy Stack Table

We can summarize which decisions are affected by which policies in a **policy stack** (constraint matrix).

**What this code does:**
- Builds a small table with rows = decisions (shifts) and columns = policy categories.
- Marks each cell with ✓ if the policy applies to that decision.

**What to look for in the output:**
- Which shifts participate in each policy.
- A visual sense of where policy complexity is concentrated.

In [4]:
# Build a simple policy stack

stack = pd.DataFrame(
    index=shifts,
    columns=["Coverage", "Night dependency", "Senior lead"],
    data=""
)

# Coverage applies to all staffed shifts
stack.loc["Day", "Coverage"] = "✓"
stack.loc["Evening", "Coverage"] = "✓"
stack.loc["Night", "Coverage"] = "✓"

# Night dependency: only Night and Evening involved
stack.loc["Evening", "Night dependency"] = "✓ (prereq)"
stack.loc["Night", "Night dependency"] = "✓ (depends)"

# Senior lead requirement: only Night involved in this example
stack.loc["Night", "Senior lead"] = "✓"

print("POLICY STACK (decisions × policy categories):")
print(stack)

POLICY STACK (decisions × policy categories):
        Coverage Night dependency Senior lead
Day            ✓                             
Evening        ✓       ✓ (prereq)            
Night          ✓      ✓ (depends)           ✓


## Step 5: Base Model – Feasible Policy Set

We now build an optimization model that **satisfies all three policies** and confirm it is feasible.

**Decisions**:
- `x_Day`, `x_Evening`, `x_Night` – 1 if the shift is staffed, 0 if not.
- `y_Night_active` – indicator: 1 if the night shift is staffed, 0 otherwise.

**Constraints**:
1. Coverage: `x_Day + x_Evening + x_Night ≥ 2`.
2. Night dependency: `x_Night ≤ x_Evening` (night can only be 1 if evening is 1).
3. Senior lead requirement: if night is staffed (`y_Night_active = 1`), a senior lead is counted in the cost; we model this with an indicator and a simple cost term.

**Objective**:
- Minimize total staffing cost subject to the rules.

**What this code does:**
- Builds the model, solves it, and prints the chosen shifts.

**What to look for in the output:**
- A valid combination of shifts that staffs at least two shifts and respects the dependency rules.

In [5]:
# Base model with feasible policies

base_model = LpProblem("Shift_Policies_Base", LpMinimize)

# Decision variables for shifts
x = {s: LpVariable(f"x_{s}", lowBound=0, upBound=1, cat="Binary") for s in shifts}

# Indicator for night shift active (simple copy of x_Night in this example)
y_Night_active = LpVariable("y_Night_active", lowBound=0, upBound=1, cat="Binary")

# Objective: minimize staffing + simple senior lead cost when night is active
base_model += (
    lpSum(costs[s] * x[s] for s in shifts) + 3 * y_Night_active,
    "Total_Cost"
)

# Coverage: at least two shifts staffed
base_model += lpSum(x[s] for s in shifts) >= 2, "Coverage_at_least_two_shifts"

# Night dependency: night can only run if evening is staffed
base_model += x["Night"] <= x["Evening"], "Night_depends_on_Evening"

# Indicator tie: y_Night_active = x_Night
base_model += y_Night_active == x["Night"], "Indicator_equals_x_Night"

solver = PULP_CBC_CMD(msg=False)
base_model.solve(solver)

status_base = LpStatus[base_model.status]
print("Base model status:", status_base)

base_solution = pd.DataFrame(
    {
        "Shift": shifts,
        "x (staffed 0/1)": [int(x[s].value()) for s in shifts],
        "Cost if staffed": [costs[s] for s in shifts],
    }
)
base_solution["Cost contribution"] = base_solution["x (staffed 0/1)"] * base_solution["Cost if staffed"]

print("\nBASE SOLUTION (feasible policies):")
print(base_solution.to_string(index=False))
print("\nNight indicator y_Night_active:", int(y_Night_active.value()))
print("Total cost:", int(base_solution["Cost contribution"].sum() + 3 * int(y_Night_active.value())))

# Validation: coverage and dependency must hold in base solution
assert base_solution["x (staffed 0/1)"].sum() >= 2, "Coverage rule violated in base solution."
if int(x["Night"].value()) == 1:
    assert int(x["Evening"].value()) == 1, "Night dependency violated in base solution."
print("\nValidation passed: base model satisfies all initial policies and is feasible.")

Base model status: Optimal

BASE SOLUTION (feasible policies):
  Shift  x (staffed 0/1)  Cost if staffed  Cost contribution
    Day                1                5                  5
Evening                1                6                  6
  Night                0                7                  0

Night indicator y_Night_active: 0
Total cost: 11

Validation passed: base model satisfies all initial policies and is feasible.


## Step 6: Add an Over-Tight Policy – Maximum 1 Shift Staffed

Now suppose a new policy is proposed:

> "To save costs, at most one shift may be staffed."

This conflicts with the existing coverage rule:
- Coverage requires **at least two** shifts.
- New rule allows **at most one** shift.

Together, these create an **impossible** requirement.

**What this code does:**
- Creates a copy of the base model and adds the new policy:
  - `x_Day + x_Evening + x_Night ≤ 1`
- Attempts to solve and reports the status.

**What to look for in the output:**
- The solver status should be `Infeasible`.

In [6]:
# Infeasible model with over-tight policy: at most one shift staffed

infeasible_model = LpProblem("Shift_Policies_Infeasible", LpMinimize)

x_inf = {s: LpVariable(f"x_{s}", lowBound=0, upBound=1, cat="Binary") for s in shifts}

infeasible_model += lpSum(costs[s] * x_inf[s] for s in shifts), "Total_Cost"

# Original coverage rule (at least two)
infeasible_model += lpSum(x_inf[s] for s in shifts) >= 2, "Coverage_at_least_two_shifts"

# New over-tight rule: at most one shift
infeasible_model += lpSum(x_inf[s] for s in shifts) <= 1, "At_most_one_shift"

infeasible_model.solve(solver)

status_inf = LpStatus[infeasible_model.status]
print("Infeasible model status:", status_inf)

assert status_inf == "Infeasible", "Model should be infeasible with conflicting coverage rules."
print("\nValidation passed: conflicting policies make the model infeasible, as expected.")

Infeasible model status: Infeasible

Validation passed: conflicting policies make the model infeasible, as expected.


## Step 7: Diagnosing the Conflict by Relaxing Policies

When a model is infeasible, you can **temporarily relax** policies to see which one restores feasibility.

**What this code does:**
- Builds two diagnostic models:
  1. Relax coverage (remove `≥ 2`, keep `≤ 1`).
  2. Relax the "at most one" rule (keep `≥ 2`, remove `≤ 1`).
- Solves each and reports whether it is feasible.

**What to look for in the output:**
- Feasibility returns when **either** coverage or the "at most one" rule is removed.
- This shows that the **pair** of rules is in conflict.

In [7]:
# Diagnostic model 1: relax coverage, keep "at most one"

model_relax_coverage = LpProblem("Relax_Coverage_Keep_AtMostOne", LpMinimize)

x_rc = {s: LpVariable(f"x_{s}", lowBound=0, upBound=1, cat="Binary") for s in shifts}

model_relax_coverage += lpSum(costs[s] * x_rc[s] for s in shifts), "Total_Cost"

# Only keep at most one rule
model_relax_coverage += lpSum(x_rc[s] for s in shifts) <= 1, "At_most_one_shift"

model_relax_coverage.solve(solver)
status_rc = LpStatus[model_relax_coverage.status]
print("Diagnostic 1 status (relax coverage, keep <=1):", status_rc)

# Diagnostic model 2: keep coverage, relax "at most one"

model_relax_atmost = LpProblem("Keep_Coverage_Relax_AtMostOne", LpMinimize)

x_ra = {s: LpVariable(f"x_{s}", lowBound=0, upBound=1, cat="Binary") for s in shifts}

model_relax_atmost += lpSum(costs[s] * x_ra[s] for s in shifts), "Total_Cost"

# Only keep coverage rule
model_relax_atmost += lpSum(x_ra[s] for s in shifts) >= 2, "Coverage_at_least_two_shifts"

model_relax_atmost.solve(solver)
status_ra = LpStatus[model_relax_atmost.status]
print("Diagnostic 2 status (keep coverage, relax <=1):", status_ra)

assert status_rc == "Optimal" and status_ra == "Optimal", (
    "Both diagnostic models should be feasible once one conflicting rule is relaxed."
)
print("\nValidation passed: relaxing either rule restores feasibility, confirming their conflict.")

Diagnostic 1 status (relax coverage, keep <=1): Optimal
Diagnostic 2 status (keep coverage, relax <=1): Optimal

Validation passed: relaxing either rule restores feasibility, confirming their conflict.


## Conclusion: Using Policy Stacks and Infeasibility as a Diagnostic Tool

In this notebook, you saw that:

- A **policy stack** summarizes which decisions are subject to which rules.
- The base model satisfied all initial rules and produced a feasible solution.
- Adding a new cost-cutting rule ("at most one shift") directly conflicted with the coverage rule ("at least two shifts").
- The solver reported **infeasible**, correctly indicating no solution exists under the combined policies.
- By relaxing one rule at a time, you identified the conflicting pair – a process that mirrors real-world policy diagnosis.

In practice, infeasibility is a **starting point for a policy conversation**, not the end of modeling.